# BPE training example
练手熟悉bpe训练过程。

In [20]:
import os
import re


vocab: dict[int,bytes]
merges: list[tuple[bytes, bytes]] = []

corpus = """low low low low low
lower lower widest widest widest
newest newest newest newest newest newest"""

vocab = {i:bytes([i]) for i in range(256)}
vocab[len(vocab)] = "<|endoftext|>".encode("utf-8")

for i in range(250,257):
    print(vocab[i])

b'\xfa'
b'\xfb'
b'\xfc'
b'\xfd'
b'\xfe'
b'\xff'
b'<|endoftext|>'


先做pretokenization，将文本粗粒度的分块。

这样一是更有利于统计byte pair的出现频率，比如对于"low"这个pretoken，如果"low"出现了5次，其中的pair byte，b"lo"和"ow"也相应地出现了5次。假如我们不做pretokenization，每一次的merge都需要我们先遍历整个corpus的所有byte pair统计频率，然后再遍历一遍做merge。而做了pretokenization之后，我们每次都只需要遍历pretoken的频率词典就好，其中的每个词都相当于被复用了。

二是有助于防止不应该的merge。比如“We have an apple!"，中间夹着空格的'n a'这样跨越边界的词很难说有什么意义。对于标点符号的区分也有所帮助，"apple!"和"apple."的意义应该是相近的，如果不做pretokenization，可能会被合并为两个完全不同的token。

所以根据课件，现在的LLM一般使用的是基于regex的pretokenizer，下面的regex来自GPT-2：
```python
>>>PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
```
It may be useful to interactively split some text with this pre-tokenizer to get a better sense of its
behavior:
```python
>>> # requires `regex` package
>>> import regex as re
>>> re.findall(PAT, "some text that i'll pre-tokenize")
['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']
```

可以看到大致的表现是根据空白进行拆分，前置空格，拆出像"'ll","'re"这些英文中比较特殊的缩写，标点符号也被单独拆出。

In [21]:
pretokens = corpus.split()
pretoken_freq: dict[tuple[bytes,...], int] = {}

for pretoken in pretokens:
    pretoken_bytes = pretoken.encode("utf-8")
    key = tuple(pretoken_bytes[i:i+1] for i in range(len(pretoken_bytes)))
    if key in pretoken_freq:
        pretoken_freq[key] += 1
    else:
        pretoken_freq[key] = 1
    

print(pretoken_freq)

{(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'w', b'i', b'd', b'e', b's', b't'): 3, (b'n', b'e', b'w', b'e', b's', b't'): 6}


In [22]:
merge_count = 6

def merge_pretoken(pretoken:tuple[bytes,...], to_merge:tuple[bytes, bytes]):
    i = 0
    result = []
    while i < len(pretoken) - 1:
        if pretoken[i:i+2] == to_merge:
            result.append(to_merge[0] + to_merge[1])
            i += 2
        else:
            result.append(pretoken[i])
            i += 1
    if i == len(pretoken) - 1:
        result.append(pretoken[i])
    
    return tuple(result)
        

for i in range(merge_count):
    # 统计所有bytes pair的频率
    pair_bytes_freq: dict[tuple[bytes, bytes], int] = {}
    for pretoken, count in pretoken_freq.items():
        for j in range(len(pretoken) - 1):
            key = pretoken[j:j+2]
            if key in pair_bytes_freq:
                pair_bytes_freq[key] += count
            else:
                pair_bytes_freq[key] = count

    # 找到频率最大（平局按字典序）的bytes pair
    max_count = max(pair_bytes_freq.values())
    to_merge = max([k for k,v in pair_bytes_freq.items() if v == max_count])

    # 更新merges和vocab
    merges.append(to_merge)
    vocab[len(vocab)] = to_merge[0] + to_merge[1]

    # 将pretoken_freq中的key按照to_merge进行合并
    new_dict:dict[tuple[bytes,...], int] = {merge_pretoken(k, to_merge):v for k, v in pretoken_freq.items()}
    pretoken_freq = new_dict

for i in range(len(vocab) - 1, 255, -1):
    print(vocab[i])




    


b'ne'
b'west'
b'low'
b'ow'
b'est'
b'st'
b'<|endoftext|>'
